# L4 - Evaluate

Decoding is tuned on **validation**. Test is decoded **once**, at the end, with
whatever validation chose. Touching test twice turns it into a second
validation set and the number stops meaning anything.

Reported against the copy baseline, never alone.

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/Amay-M-Nair/AttentionMech.git"

def has_src(d):
    return os.path.isdir(os.path.join(d, "src"))

if not any(has_src(d) for d in (os.getcwd(), os.path.dirname(os.getcwd()))):
    target = os.path.join(os.getcwd(), "AttentionMech")
    if os.path.isdir(os.path.join(target, ".git")):
        subprocess.run(["git", "-C", target, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO, target], check=True)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "sentencepiece", "sacrebleu"], check=True)

print("cwd:", os.getcwd())

In [ ]:
import time

import matplotlib.pyplot as plt

from src.config import get_device
from src.dataset import load_split
from src.evaluate import bleu, chrf, copy_baseline
from src.inference import translate_corpus
from src.spm_tokenizer import SPMTokenizer
from src.train import load_checkpoint

%matplotlib inline
for candidate in (os.getcwd(), os.path.dirname(os.getcwd())):
    if has_src(candidate):
        ROOT = candidate
        break

DATA = os.path.join(ROOT, "data")
CHECKPOINT = os.path.join(ROOT, "checkpoints", "latin.pt")
MAX_LEN, GEN_CAP = 128, 200

device = get_device()
tokenizer = SPMTokenizer(os.path.join(DATA, "spm8k.model"))

valid_la, valid_en = load_split(DATA, "valid")
test_la, test_en = load_split(DATA, "test")
test_src = open(os.path.join(DATA, "test.src"), encoding="utf-8").read().splitlines()

valid_baseline = copy_baseline(valid_la, valid_en)
test_baseline = copy_baseline(test_la, test_en)

model, meta = load_checkpoint(CHECKPOINT, device=device)
print(f"checkpoint: epoch {meta['epoch']}, valid BLEU {meta['score']:.2f}")
print(f"valid {len(valid_la):,} | test {len(test_la):,}")
print(f"copy baseline   valid {valid_baseline:.2f}   test {test_baseline:.2f}")

## Tuning on validation

Two sweeps rather than a grid: beam size at a fixed penalty, then the penalty
at the winning beam. Ten decodes instead of twenty-five, and the two interact
weakly.

`amp=True` here for speed. The test number below is decoded in fp32.

In [ ]:
def score(beam, alpha, source=None, target=None, amp=True, batch=24):
    source = valid_la if source is None else source
    target = valid_en if target is None else target
    started = time.time()
    hyp = translate_corpus(model, source, tokenizer, device, batch_size=batch,
                           beam_size=beam, length_penalty=alpha,
                           src_max_len=MAX_LEN, max_len=GEN_CAP, amp=amp)
    return bleu(hyp, target), time.time() - started


beam_results = {}
for beam in (1, 2, 4, 6, 8):
    s, secs = score(beam, 1.0)
    beam_results[beam] = s
    print(f"beam {beam}  alpha 1.0   BLEU {s:6.2f}   ({secs/60:.1f} min)", flush=True)

best_beam = max(beam_results, key=beam_results.get)
print(f"\nbest beam {best_beam}")

In [ ]:
alpha_results = {1.0: beam_results[best_beam]}
for alpha in (0.0, 0.6, 1.5, 2.0):
    alpha_results[alpha], secs = score(best_beam, alpha)

for alpha in sorted(alpha_results):
    print(f"beam {best_beam}  alpha {alpha:3.1f}   BLEU {alpha_results[alpha]:6.2f}")

best_alpha = max(alpha_results, key=alpha_results.get)
print(f"\nchosen: beam {best_beam}, alpha {best_alpha}  ->  "
      f"valid BLEU {alpha_results[best_alpha]:.2f} (baseline {valid_baseline:.2f})")

## Test - decoded once

fp32, so the number is exactly reproducible.

In [ ]:
started = time.time()
test_hyp = translate_corpus(model, test_la, tokenizer, device, batch_size=24,
                            beam_size=best_beam, length_penalty=best_alpha,
                            src_max_len=MAX_LEN, max_len=GEN_CAP, amp=False)
print(f"decoded {len(test_hyp):,} sentences in {(time.time()-started)/60:.1f} min\n")

test_bleu = bleu(test_hyp, test_en)
test_chrf = chrf(test_hyp, test_en)
base_chrf = chrf(test_la, test_en)

print(f"{'':8} {'BLEU':>8} {'chrF':>8}")
print(f"{'copy':8} {test_baseline:8.2f} {base_chrf:8.2f}")
print(f"{'model':8} {test_bleu:8.2f} {test_chrf:8.2f}")
print(f"\n{test_bleu/test_baseline:.0f}x the baseline")

## Where it works

By source length, and Vulgate against everything else. The second is the
interesting one - a third of the corpus is formulaic scripture and the rest is
real prose.

In [ ]:
lengths = [len(tokenizer.encode(line)) for line in test_la]
buckets = [("1-20", 1, 20), ("21-35", 21, 35), ("36-55", 36, 55), ("56+", 56, 10**6)]

def subset(idx):
    h = [test_hyp[i] for i in idx]
    r = [test_en[i] for i in idx]
    return bleu([test_la[i] for i in idx], r), bleu(h, r), chrf(h, r)

print(f"{'src tokens':>12} {'n':>6} {'copy':>7} {'BLEU':>7} {'chrF':>7}")
length_rows = []
for name, lo, hi in buckets:
    idx = [i for i, n in enumerate(lengths) if lo <= n <= hi]
    if not idx:
        continue
    base, b, c = subset(idx)
    length_rows.append((name, len(idx), base, b, c))
    print(f"{name:>12} {len(idx):>6} {base:7.2f} {b:7.2f} {c:7.2f}")

In [ ]:
vulgate = [i for i, s in enumerate(test_src) if s == "Vulgate_Bible"]
classical = [i for i, s in enumerate(test_src) if s != "Vulgate_Bible"]

print(f"{'':12} {'n':>6} {'copy':>7} {'BLEU':>7} {'chrF':>7}")
split_rows = []
for name, idx in (("Vulgate", vulgate), ("classical", classical)):
    base, b, c = subset(idx)
    split_rows.append((name, len(idx), base, b, c))
    print(f"{name:12} {len(idx):>6} {base:7.2f} {b:7.2f} {c:7.2f}")

print(f"\n{len(set(test_src))} distinct works in test; "
      f"{len(vulgate)/len(test_src):.1%} Vulgate")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

axes[0].bar([r[0] for r in length_rows], [r[3] for r in length_rows], color="tab:blue")
axes[0].set_xlabel("source tokens"); axes[0].set_ylabel("BLEU")
axes[0].set_title("By length")

axes[1].bar([r[0] for r in split_rows], [r[3] for r in split_rows],
            color=["tab:green", "tab:orange"])
axes[1].set_ylabel("BLEU"); axes[1].set_title("By source")

for ax in axes:
    ax.axhline(test_bleu, color="tab:gray", ls="--", label=f"overall {test_bleu:.1f}")
    ax.legend(); ax.grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

## Read the output

In [ ]:
short_vulgate = sorted(vulgate, key=lambda i: lengths[i])[:3]
long_classical = sorted(classical, key=lambda i: lengths[i])[-3:]

for label, idx in (("VULGATE", short_vulgate), ("CLASSICAL, longest", long_classical)):
    print(f"----- {label} -----")
    for i in idx:
        print("LA  :", test_la[i][:110])
        print("PRED:", test_hyp[i][:110])
        print("REF :", test_en[i][:110])
        print()

## Done

Decoding chosen on validation, test scored once, reported beside the baseline.